In [3]:
from langchain_core.messages.utils import trim_messages, count_tokens_approximately
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

MAX_TOKENS = 150


def call_model(state: MessagesState):
    trimmed = trim_messages(
        state["messages"],
        max_tokens=MAX_TOKENS,
        strategy="last",
        token_counter=count_tokens_approximately,
        include_system=True,
        allow_partial=False,
    )
    response = llm.invoke(trimmed)
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("chat", call_model)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [4]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}


def show_state():
    msgs = graph.get_state(config).values["messages"]
    print("token count:", count_tokens_approximately(msgs))
    for m in msgs:
        print("-", type(m).__name__, ":", str(m.content)[:60])
    print("total messages:", len(msgs))

In [5]:
graph.invoke({"messages": [HumanMessage(content="Hi, my name is Aayush")]}, config=config)
show_state()

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


token count: 30
- HumanMessage : Hi, my name is Aayush
- AIMessage : [{'type': 'text', 'text': 'Hello Aayush! Nice to meet you. H
total messages: 2


In [6]:
graph.invoke({"messages": [HumanMessage(content="Explain gradient descent in detail")]}, config=config)
show_state()

token count: 1683
- HumanMessage : Hi, my name is Aayush
- AIMessage : [{'type': 'text', 'text': 'Hello Aayush! Nice to meet you. H
- HumanMessage : Explain gradient descent in detail
- AIMessage : [{'type': 'text', 'text': '**Gradient Descent** is one of th
total messages: 4


In [7]:
graph.invoke({"messages": [HumanMessage(content="What is my name?")]}, config=config)
show_state()

token count: 1712
- HumanMessage : Hi, my name is Aayush
- AIMessage : [{'type': 'text', 'text': 'Hello Aayush! Nice to meet you. H
- HumanMessage : Explain gradient descent in detail
- AIMessage : [{'type': 'text', 'text': '**Gradient Descent** is one of th
- HumanMessage : What is my name?
- AIMessage : [{'type': 'text', 'text': "I don't know your name yet! What 
total messages: 6
